# WEEK 5  – ASSIGNMENT

### Spark DataFrame Operations - Data Cleaning & Transformation

#### Objective:Understand Spark fundamentals and perform data cleaning, transformation, and aggregation using DataFrames.

##### Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing? 

Traditional MapReduce is a distributed processing framework that works well for batch processing, but it has several limitations that make Apache Spark a better choice for modern big data applications.

**Limitations of MapReduce**
- Disk based processing: Writes intermediate results to disk after every Map and Reduce phase.
- Slow performance: Frequent disk reads and writes increase execution time. 
- Not suitable for iterative algorithms: Reloads data from disk in every iteration, making machine learning and graph processing inefficient. 
- Complex programming model: Requires writing separate Map and Reduce functions, leading to more code and higher development effort. 
- No real-time processing: Designed mainly for batch processing; not suitable for interactive or streaming applications. 
- Limited built-in libraries: Does not provide native support for SQL, machine learning, graph processing, or stream processing

**Why Spark is Preferred**
- In-memory computing: Stores intermediate data in memory, reducing disk I/O. 
- Faster execution: Can be up to 100× faster in memory and 10× faster on disk than MapReduce for many workloads. 
- Efficient for iterative processing: Reuses data in memory, making machine learning and graph algorithms much faster. 
- Easy to use APIs: Provides simple APIs in Python, Scala, Java, and R, along with DataFrames and Spark SQL. 
- Supports multiple workloads: Handles batch processing, real-time streaming, SQL queries, machine learning, and graph processing in a single framework. 
- Optimized execution: Uses lazy evaluation and query optimization to improve performance.


##### Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

- In-memory computing means Spark stores intermediate data in RAM (memory) instead of writing it to disk after every operation. 
- Machine learning algorithms are iterative, meaning they repeatedly process the same dataset multiple times to improve the model. 
- In disk-based systems like MapReduce, intermediate results are written to disk and read back in every iteration, causing high disk I/O and slower execution. 
- Spark caches frequently used datasets in memory using operations like cache() or persist(). 
- Since the data is already available in memory, Spark avoids repeated disk reads and writes, significantly reducing processing time. 
- This makes iterative algorithms such as Linear Regression, Logistic Regression, K-Means Clustering, and Decision Trees much faster. 
- Spark's in-memory processing also improves performance for interactive data analysis and real-time applications. 
- As a result, Spark can be up to 100× faster in memory and around 10× faster on disk than traditional MapReduce for many workloads.

#### Load Dataset

In [0]:
df = spark.read.csv("/Volumes/workspace/default/my_volume/sales_records.csv",header=True,inferSchema=True)

In [0]:
df.printSchema()

root
 |-- record_id: integer (nullable = true)
 |-- user_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- status: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: string (nullable = true)
 |-- raw_timestamp: string (nullable = true)



In [0]:
df.display()

record_id,user_id,transaction_date,region,product_category,sale_amount,city,age,subscription,email,username,status,price,store_id,raw_timestamp
1,U0064,2023-06-04,East,Sports,775.05,Denver,54,Free,user174@example.com,user_147,Inactive,150.07,S003,07-05-2023 15:52
2,U0126,2023-02-09,East,Food,1902.33,Seattle,53,Basic,user269@example.com,user_143,Active,356.85,S013,15-05-2023 22:57
3,U0025,2023-12-31,North,Food,324.77,Portland,41,Free,user14@example.com,user_75,Inactive,147.79,S028,24-11-2023 18:07
4,U0073,2023-09-23,West,Food,424.46,Seattle,65,Free,user279@example.com,user_86,Unknown,171.45,S010,05-02-2023 02:04
5,U0028,2023-05-24,North,Food,687.55,Denver,28,Free,user170@example.com,user_157,Pending,67.02,S020,21-12-2023 16:32
6,U0031,2023-10-26,East,Clothing,1835.12,Los Angeles,16,Premium,user159@example.com,user_197,Active,220.4,S004,15-07-2023 06:53
7,U0067,2023-02-28,South,Food,797.71,Seattle,32,Basic,user142@example.com,user_13,Active,121.56,S010,23-11-2023 05:54
8,U0099,2023-07-11,North,Food,1688.88,San Francisco,23,Premium,user297@example.com,user_46,null,79.1,S015,11-05-2023 05:09
9,U0082,2023-09-26,West,Electronics,1153.59,San Francisco,49,Premium,user254@example.com,user_60,Inactive,277.32,S019,20-03-2023 15:06
10,U0131,2023-07-31,West,Books,1476.31,Los Angeles,41,Basic,null,user_76,Pending,29.1,S007,02-10-2023 20:11


In [0]:
df.count()

600

##### Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date. 

In [0]:
print("Total rows before removing duplicates:",df.count())

Total rows before removing duplicates: 600


In [0]:
df_deduped=df.dropDuplicates(["user_id","transaction_date"])
print("Total rows after removing duplicates:",df_deduped.count())

Total rows after removing duplicates: 518


##### Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

In [0]:
# Renamed df_deduped to df_sales
df_sales=df
print("df_sales row count:",df_sales.count())

df_sales row count: 600


In [0]:
from pyspark.sql.functions import col,avg, round
df_result = (
df_sales
.filter(col("region") == "West")
.groupBy("product_category")
.agg(round(avg("sale_amount"), 2).alias("avg_sale_amount"))
.orderBy("avg_sale_amount", ascending=False)
)
df_result.display()

product_category,avg_sale_amount
Books,1091.05
Clothing,1025.11
Electronics,1023.6
Food,969.17
Sports,943.0


##### Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'. 

Difference between .na.drop() and .na.fill()

1. .na.drop()
   - Removes rows that contain null values.
   - Used when missing values should be discarded.

2. .na.fill()
   - Replaces null values with a specified value.
   - Used when you want to keep all rows and fill missing data.

In [0]:
#Demonstrate .na.drop()
from pyspark.sql.functions import col, sum as spark_sum
null_count = df.filter(col("status").isNull()).count()
print("Null values in status column Before fill:", null_count)

Null values in status column BEFORE fill: 74


In [0]:

df_dropped = df.na.drop(subset=["status"])
print("Total rows Before drop:", df.count())
print("Total rows After drop:", df_dropped.count())
print("Rows removed:", df.count() - df_dropped.count())

Total rows Before drop: 600
Total rows After drop: 526
Rows removed: 74


In [0]:
#Demonstrate .na.fill()
#Replacing  null values with a given value unknown
df_filled = df.na.fill({"status": "Unknown"})
null_before=df.filter(col("status").isNull()).count()
null_after = df_filled.filter(col("status").isNull()).count()
print("Null values in status column Before fill:",null_before)
print("Null values in status column After fill:", null_after)

Null values in status column Before fill: 74
Null values in status column After fill: 0


##### Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100. 

In [0]:
from pyspark.sql.functions import count, col
df_city_count = (
df.groupBy("city")
.agg(count("*").alias("total_records"))
.filter(col("total_records")>100)
.orderBy("total_records", ascending=False)
)

df_city_count.display()

city,total_records
Los Angeles,178
Seattle,162
San Francisco,144


##### Q7: How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them? 

Spark DataFrames are immutable, which means once a DataFrame is created, it cannot be modified directly. Any operation such as dropping a column, renaming a column, or filtering rows creates a new DataFrame, while the original DataFrame remains unchanged.
This makes data processing more reliable because the original data is preserved and transformations can be chained together safely.

##### Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'. 

In [0]:
from pyspark.sql.functions import col
df_filtered = (
df.filter(
(col("age").between(18, 30)) &
(col("subscription") == "Premium"))
)
df_filtered.display()

record_id,user_id,transaction_date,region,product_category,sale_amount,city,age,subscription,email,username,status,price,store_id,raw_timestamp
8,U0099,2023-07-11,North,Food,1688.88,San Francisco,23,Premium,user297@example.com,user_46,null,79.1,S015,11-05-2023 05:09
33,U0061,2023-07-25,West,Sports,328.8,Los Angeles,23,Premium,user47@example.com,user_150,Pending,412.22,S028,11-04-2023 04:35
56,U0098,2023-05-02,West,Books,257.24,Seattle,21,Premium,user232@example.com,user_47,Inactive,375.85,S005,13-04-2023 02:07
60,U0052,2023-09-10,South,Books,595.22,Tucson,30,Premium,user84@example.com,user_253,Inactive,286.53,S004,12-05-2023 01:21
64,U0136,2023-07-21,North,Sports,876.96,Seattle,26,Premium,user203@example.com,user_274,Active,469.72,S008,08-01-2023 23:33
69,U0136,2023-07-21,North,Sports,876.96,Seattle,26,Premium,user203@example.com,user_274,Active,469.72,S008,08-01-2023 23:33
87,U0001,2023-04-27,East,Sports,null,Los Angeles,19,Premium,null,user_287,Pending,136.8,S010,16-01-2023 19:40
119,U0107,2023-02-04,North,Food,349.31,Seattle,23,Premium,user106@example.com,user_166,Unknown,412.64,S005,16-12-2023 20:53
120,U0122,2023-01-20,West,Sports,1897.96,Los Angeles,20,Premium,user93@example.com,user_299,Active,160.49,S003,12-10-2023 13:32
123,U0045,2023-11-07,West,Food,312.25,Los Angeles,26,Premium,user286@example.com,user_202,null,166.62,S019,17-08-2023 21:33


##### Q9: When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?

Null values should be handled before performing mathematical aggregations like sum() or avg() because they can affect the accuracy and reliability of the results. Missing values may cause incorrect calculations or lead to incomplete analysis if they are not cleaned first.
- If a value is missing, the average may not accurately represent the entire dataset.
- Replacing null values or removing them ensures that calculations are performed on valid data.
- Cleaning null values improves the quality and consistency of the analysis.

##### Q10: Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time. 

In [0]:
from pyspark.sql.functions import col, to_timestamp
df_updated = (
df.withColumn("event_time",to_timestamp(col("raw_timestamp"), "dd-MM-yyyy HH:mm")).drop("raw_timestamp")
)
display(df_updated.limit(10))

record_id,user_id,transaction_date,region,product_category,sale_amount,city,age,subscription,email,username,status,price,store_id,event_time
1,U0064,2023-06-04,East,Sports,775.05,Denver,54,Free,user174@example.com,user_147,Inactive,150.07,S003,2023-05-07T15:52:00.000Z
2,U0126,2023-02-09,East,Food,1902.33,Seattle,53,Basic,user269@example.com,user_143,Active,356.85,S013,2023-05-15T22:57:00.000Z
3,U0025,2023-12-31,North,Food,324.77,Portland,41,Free,user14@example.com,user_75,Inactive,147.79,S028,2023-11-24T18:07:00.000Z
4,U0073,2023-09-23,West,Food,424.46,Seattle,65,Free,user279@example.com,user_86,Unknown,171.45,S010,2023-02-05T02:04:00.000Z
5,U0028,2023-05-24,North,Food,687.55,Denver,28,Free,user170@example.com,user_157,Pending,67.02,S020,2023-12-21T16:32:00.000Z
6,U0031,2023-10-26,East,Clothing,1835.12,Los Angeles,16,Premium,user159@example.com,user_197,Active,220.4,S004,2023-07-15T06:53:00.000Z
7,U0067,2023-02-28,South,Food,797.71,Seattle,32,Basic,user142@example.com,user_13,Active,121.56,S010,2023-11-23T05:54:00.000Z
8,U0099,2023-07-11,North,Food,1688.88,San Francisco,23,Premium,user297@example.com,user_46,null,79.1,S015,2023-05-11T05:09:00.000Z
9,U0082,2023-09-26,West,Electronics,1153.59,San Francisco,49,Premium,user254@example.com,user_60,Inactive,277.32,S019,2023-03-20T15:06:00.000Z
10,U0131,2023-07-31,West,Books,1476.31,Los Angeles,41,Basic,null,user_76,Pending,29.1,S007,2023-10-02T20:11:00.000Z


##### Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation? 

Shuffle is the process in Apache Spark where data is redistributed across partitions so that records with the same key are brought together. This commonly happens during operations like groupBy(),join(),distinct(),and reduceByKey().

For example, when you perform groupBy("city"), Spark moves all records belonging to the same city into the same partition before calculating the results.

It is considered a wide transformation because data is transferred between different partitions and executors. Unlike narrow transformations, a wide transformation requires network communication, making it more expensive in terms of time and resources.

##### Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

In [0]:
from pyspark.sql.functions import col
df_clean = df.filter( col("email").isNotNull() &(col("username") != ""))
display(df_clean.limit(10))

record_id,user_id,transaction_date,region,product_category,sale_amount,city,age,subscription,email,username,status,price,store_id,raw_timestamp
1,U0064,2023-06-04,East,Sports,775.05,Denver,54,Free,user174@example.com,user_147,Inactive,150.07,S003,07-05-2023 15:52
2,U0126,2023-02-09,East,Food,1902.33,Seattle,53,Basic,user269@example.com,user_143,Active,356.85,S013,15-05-2023 22:57
3,U0025,2023-12-31,North,Food,324.77,Portland,41,Free,user14@example.com,user_75,Inactive,147.79,S028,24-11-2023 18:07
4,U0073,2023-09-23,West,Food,424.46,Seattle,65,Free,user279@example.com,user_86,Unknown,171.45,S010,05-02-2023 02:04
5,U0028,2023-05-24,North,Food,687.55,Denver,28,Free,user170@example.com,user_157,Pending,67.02,S020,21-12-2023 16:32
6,U0031,2023-10-26,East,Clothing,1835.12,Los Angeles,16,Premium,user159@example.com,user_197,Active,220.4,S004,15-07-2023 06:53
7,U0067,2023-02-28,South,Food,797.71,Seattle,32,Basic,user142@example.com,user_13,Active,121.56,S010,23-11-2023 05:54
8,U0099,2023-07-11,North,Food,1688.88,San Francisco,23,Premium,user297@example.com,user_46,null,79.1,S015,11-05-2023 05:09
9,U0082,2023-09-26,West,Electronics,1153.59,San Francisco,49,Premium,user254@example.com,user_60,Inactive,277.32,S019,20-03-2023 15:06
11,U0135,2023-02-14,East,Clothing,684.85,Seattle,64,Premium,user59@example.com,user_239,Active,148.93,S027,03-01-2023 11:38


##### Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

In [0]:
from pyspark.sql.functions import min, max, mean
df.agg(
min("price").alias("min_price"),
max("price").alias("max_price"),
mean("price").alias("mean_price")
).display()

min_price,max_price,mean_price
1.27,499.93,250.4618053097347


##### Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?

When using inferSchema=true, Spark automatically determines the data type of each column by examining the data. If the source data contains messy or inconsistent date formats, Spark may infer the wrong data type or fail to parse some values correctly.

Risks
- Date columns may be inferred as StringType instead of DateType or TimestampType.
- Some date values may become NULL if Spark cannot parse them.
- Incorrect schema inference can affect filtering, sorting, joins, and date based calculations.

##### Q15: Write a final processing pipeline that: 

Filters out duplicates. 

Fills null prices with 0. 

Groups by store_id to calculate total revenue. 

In [0]:
from pyspark.sql.functions import sum
df_result = (
df.dropDuplicates()
.na.fill({"price": 0})
.groupBy("store_id")
.agg(sum("price").alias("total_revenue"))
.orderBy("total_revenue", ascending=False)
)
df_result.display()

store_id,total_revenue
S015,7802.729999999998
S028,6372.51
S019,6320.670000000001
S014,6135.2300000000005
S005,5948.050000000001
S011,5706.219999999999
S003,5612.319999999998
S022,5531.2
S024,5359.88
S023,5208.75
